# Explore a .bcmodel file

Interactive notebook for loading, inspecting, and visualizing BrainChop models.

## Prerequisites

```bash
cd bcmodel-rs/bcmodel-py
python -m venv .venv && source .venv/bin/activate
pip install numpy maturin matplotlib
maturin develop
```

In [ ]:
import bcmodel
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

MODEL_PATH = "../../meshnet/model5_gw_ae/model.bcmodel"

## 1. Load model and print summary

In [ ]:
model = bcmodel.BcmodelFile.load(MODEL_PATH)
print(model)
print()
print(f"Name:         {model.name}")
print(f"Type:         {model.model_type}")
print(f"Description:  {model.description}")
print(f"Input shape:  {model.input_shape}")
print(f"Classes:      {model.num_classes}")
print(f"Graph nodes:  {model.graph_length}")
print(f"Parameters:   {model.total_params:,}")
print(f"Weight size:  {model.weight_size_bytes / 1024:.1f} KB")

## 2. Architecture graph

In [ ]:
graph = model.graph()

# Print the graph as a table
print(f"{'ID':20s} {'Op':15s} {'Inputs'}")
print("-" * 60)
for node in graph:
    inputs = ", ".join(node["inputs"]) if node["inputs"] else "(model input)"
    print(f"{node['id']:20s} {node['op']:15s} {inputs}")

In [ ]:
# Visualize operation type distribution
from collections import Counter

op_counts = Counter(node["op"] for node in graph)

fig, ax = plt.subplots(figsize=(8, 3))
ops = list(op_counts.keys())
counts = list(op_counts.values())
colors = plt.cm.Set2(np.linspace(0, 1, len(ops)))

ax.barh(ops, counts, color=colors)
ax.set_xlabel("Count")
ax.set_title("Operations in model graph")
for i, c in enumerate(counts):
    ax.text(c + 0.1, i, str(c), va="center")
plt.tight_layout()
plt.show()

## 3. Weight tensors

In [ ]:
# List all tensors with shapes and stats
print(f"{'Tensor':30s} {'Shape':25s} {'Min':>10s} {'Max':>10s} {'Mean':>10s} {'Std':>10s}")
print("-" * 95)

for name in model.tensor_names():
    shape = model.tensor_shape(name)
    t = model.tensor(name)
    print(
        f"{name:30s} {str(shape):25s} {t.min():10.4f} {t.max():10.4f} "
        f"{t.mean():10.4f} {t.std():10.4f}"
    )

In [ ]:
# Plot weight distributions for convolution kernels
conv_tensors = [
    name for name in model.tensor_names() if name.endswith(".weight") and "conv" in name
]

if conv_tensors:
    fig, axes = plt.subplots(1, len(conv_tensors), figsize=(4 * len(conv_tensors), 3))
    if len(conv_tensors) == 1:
        axes = [axes]

    for ax, name in zip(axes, conv_tensors):
        t = model.tensor(name)
        ax.hist(t, bins=50, alpha=0.7, edgecolor="black", linewidth=0.5)
        ax.set_title(name, fontsize=9)
        ax.set_xlabel("Value")
        ax.set_ylabel("Count")

    plt.suptitle("Convolution weight distributions", fontsize=12)
    plt.tight_layout()
    plt.show()

## 4. Labels and colormap

In [ ]:
labels = model.labels()

if labels:
    fig, ax = plt.subplots(figsize=(6, max(2, 0.5 * len(labels))))
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, len(labels) - 0.5)
    ax.invert_yaxis()
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels([f"[{l['index']}] {l['name']}" for l in labels])
    ax.set_xticks([])
    ax.set_title("Label colormap")

    for i, label in enumerate(labels):
        r, g, b = [c / 255 for c in label["color"]]
        rect = FancyBboxPatch(
            (0.05, i - 0.4), 0.9, 0.8,
            boxstyle="round,pad=0.02",
            facecolor=(r, g, b),
            edgecolor="gray",
            linewidth=0.5,
        )
        ax.add_patch(rect)

    plt.tight_layout()
    plt.show()
else:
    print("No labels defined in this model.")

## 5. Inference configuration

In [ ]:
config = model.inference_config()
if config:
    for key, value in config.items():
        print(f"  {key:25s} = {value}")
else:
    print("No inference configuration.")

## 6. Using the compatibility API

Drop-in replacement for the pure-Python `load_bcmodel.py`:

In [ ]:
header, tensors = bcmodel.load_bcmodel(MODEL_PATH)

print(f"Format version: {header['bcmodel_version']}")
print(f"Metadata name:  {header['metadata']['name']}")
print(f"Tensor count:   {len(tensors)}")
print()

# Tensors are already reshaped numpy arrays
for name, arr in tensors.items():
    print(f"  {name:30s} shape={arr.shape}  dtype={arr.dtype}")